In [8]:
import os
from dotenv import load_dotenv
from openrouter import OpenRouter
load_dotenv(r"D:\GEN AI PROJECTS\.venv")
client=os.getenv("open_router")

In [11]:
from ipywidgets import FileUpload
from IPython.display import display

def audio_input():

    uploader = FileUpload(
        accept=".wav,.mp3,.m4a",
        multiple=False
    )

    display(uploader)

    print("Upload an audio file...")

    while len(uploader.value) == 0:
        pass

    uploaded_file = list(uploader.value.values())[0]

    audio_bytes = uploaded_file["content"]
    filename = uploaded_file["name"]

    with open(filename, "wb") as f:
        f.write(audio_bytes)

    print(f"Uploaded: {filename}")

    return filename

In [ ]:
audio_path = audio_input()

In [12]:
def audio_to_text(audio_path):
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file
        )
    return transcript.text

In [14]:
meeting_transcript = audio_to_text(audio_path)
print("Transcript:\n")
print(meeting_transcript)

NameError: name 'audio_path' is not defined

In [ ]:
from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import SystemMessage, HumanMessage

def summarize_meeting(meeting_transcript):

    chat = ChatOpenRouter(
        client=client,
        model="nvidia/nemotron-3-super-120b-a12b",
        temperature=0.3
    )

    messages = [
        SystemMessage(
            content="""
You are an expert AI Meeting Assistant.

Generate professional meeting notes.

Return:

# Meeting Summary

## Overview

## Objectives

## Key Discussion Points

## Decisions Made

## Action Items

| Task | Owner | Deadline |

If information is missing write "Not Mentioned".

## Risks / Blockers

## Important Dates

## Next Steps

## Conclusion

Do not hallucinate.
"""
        ),

        HumanMessage(
            content=meeting_transcript
        )
    ]

    response = chat.invoke(messages)

    return response.content

In [ ]:
summary = summarize_meeting(meeting_transcript)
print(summary)

In [ ]:
def question_asked(summary):
    chat = ChatOpenRouter(
        client=client,
        model="nvidia/nemotron-3-super-120b-a12b",
        temperature=0.3
    )

    messages = [
        SystemMessage("You are an expert AI Meeting Assistant. You will answer questions based on the meeting summary provided."),
        HumanMessage(
            content=summary)
    ]
    response = chat.invoke(messages)
    return response.content
   

In [ ]:
def topic_detection(summary):
    chat = ChatOpenRouter(
        client=client,
        model="nvidia/nemotron-3-super-120b-a12b",
        temperature=0.3
    )
    messages = [
        SystemMessage(
            content="""
You are an expert AI Meeting Assistant.

Your task is to identify the main topics discussed in the meeting.

Rules:
- Return only the major topics.
- Merge similar topics.
- Do not explain each topic.
- Return between 3 and 10 topics.
- Output as a bullet list.
"""
        ),
        HumanMessage(
            content=f"""
Meeting Summary:

{summary}

Extract the discussion topics.
"""
        )
    ]

    response = chat.invoke(messages)
    return response.content

In [ ]:
def sentiment_analysis(summary):
    chat = ChatOpenRouter(
        client=client,
        model="nvidia/nemotron-3-super-120b-a12b",
        temperature=0.3
    )
    messages = [
        SystemMessage(
            content="""
You are an expert AI Meeting Assistant.

Your task is to analyze the sentiment of the meeting summary.

Rules:
- Return only the overall sentiment (positive, negative, or neutral).
- Do not provide any explanation.
"""
        ),
        HumanMessage(
            content=f"""
Meeting Summary:

{summary}

Analyze the sentiment.
"""
        )
    ]

    response = chat.invoke(messages)
    return response.content